# SLD Component Detection — D-FINE Inference Notebook

**Before running:** upload your trained checkpoint (`best_stg1.pth`) via the Files panel (left sidebar in Colab/Kaggle).

`best_stg1.pth` is confirmed correct — that's D-FINE's own naming convention for its best checkpoint (earlier guidance saying `best_stat.pth` was wrong).

**Two important checks before trusting results:**
- This notebook **reconstructs** the model architecture config from your training log output, since the original custom training YAML wasn't available here. If you still have that file (the one passed to `train.py -c ...`), upload it and point `CONFIG_PATH` at it in Step 3 instead — that guarantees an exact match.
- Your training log showed `remap_mscoco_category: True`. That flag is meant for real MS-COCO category IDs, not custom datasets — if detections come back with obviously wrong/scrambled class names, this is the first thing to check (would need a retrain with it set to `False`).

**Two modes:**
- Mode 1: Tiled 5×5 with overlap
- Mode 2: Tiled 4×4 with overlap (recommended for large SLD sheets)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 1 — Install D-FINE
# ══════════════════════════════════════════════════════
import os

if not os.path.exists('/content/D-FINE'):
    !git clone --depth 1 https://github.com/Peterande/D-FINE.git /content/D-FINE

%cd /content/D-FINE
!pip install -q -r requirements.txt
!pip install -q supervision Pillow

import importlib
for pkg in ['torch', 'torchvision', 'supervision']:
    m = importlib.import_module(pkg)
    print(f'ok {pkg} {getattr(m, "__version__", "")}')


/content/D-FINE
ok torch 2.11.0+cu128
ok torchvision 0.26.0+cu128
ok supervision 0.29.1


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 2 — Config
# ══════════════════════════════════════════════════════
import os

DFINE_WEIGHTS_PATH = '/content/best_stg1.pth'

NUM_CLASSES   = 30
RESOLUTION    = 640
CONFIDENCE    = 0.20
GRID_SIZE     = 4
OVERLAP       = 0.20
IOU_THRESHOLD = 0.50

# Ordered by ascending original dataset category id (id=1 -> index 0, id=2 -> index 1, ...).
# This is the standard way COCO-style loaders (incl. D-FINE's) remap possibly-gappy
# category ids to contiguous 0..num_classes-1 training labels.
#
# NOTE: your printout only showed 28 ids (1,2,3,5-26,28,29,30 -- missing id=4 and id=27),
# but the checkpoint expects 30 classes. The two placeholders below MUST be replaced with
# the correct names for id=4 and id=27, or every class after whichever one is wrong will
# be shifted by one and mislabeled. Check your dataset's categories.json / class map.
CLASS_NAMES = [
    'ACB',                                          # id=1
    'ATS',                                           # id=2
    'Ammeter',                                       # id=3
    'UNKNOWN_ID_4',                                  # id=4  <-- CONFIRM THIS NAME
    'CIRCUIT BREAKER',                               # id=5
    'CURRENT TRANSFORMER',                           # id=6
    'Circuit Breaker 2',                             # id=7
    'Contractor 1',                                  # id=8
    'DIGITAL METER',                                 # id=9
    'Digital Power Meter',                           # id=10
    'EARTH LEAKAGE RELAY',                           # id=11
    'EMS',                                           # id=12
    'Earth Fault Relay',                             # id=13
    'FUSE 1',                                        # id=14
    'FUSE 2',                                        # id=15
    'HRC FUSE WITH BLOWN FUSE INDICATOR',            # id=16
    'ISOLATOR',                                      # id=17
    'MAXIMUM DEMAND AMMETER',                        # id=18
    'Over current Relay',                            # id=19
    'PHASE INDICATOR LIGHTS',                        # id=20
    'Power Quality Meter',                           # id=21
    'RCCB',                                          # id=22
    'SELECTOR SWITCH',                               # id=23
    'SHUNT TRIP',                                    # id=24
    'SINGLE PHASE UNFUSED TAP OFF UNIT',             # id=25
    'SURGE ARRESTOR 1',                              # id=26
    'UNKNOWN_ID_27',                                 # id=27  <-- CONFIRM THIS NAME
    'TIME DELAY RELAY',                              # id=28
    'VOLTMETER',                                     # id=29
    'ZERO PHASE SEQUENCE CURRENT TRANSFORMER',       # id=30
]

assert os.path.exists(DFINE_WEIGHTS_PATH), f'D-FINE weights not found: {DFINE_WEIGHTS_PATH}'
assert len(CLASS_NAMES) == NUM_CLASSES, (
    f'CLASS_NAMES has {len(CLASS_NAMES)} entries but NUM_CLASSES={NUM_CLASSES}. '
    'Fill in the two UNKNOWN_ID placeholders above with your real class names.'
)

print('=' * 60)
print(f'D-FINE  Weights : {DFINE_WEIGHTS_PATH}')
print(f'Num Classes     : {NUM_CLASSES}')
print(f'Resolution      : {RESOLUTION}')
print(f'Confidence      : {CONFIDENCE}')
print(f'Classes         : {CLASS_NAMES}')
print('=' * 60)


D-FINE  Weights : /content/best_stg1.pth
Num Classes     : 30
Resolution      : 640
Confidence      : 0.2
Classes         : ['ACB', 'ATS', 'Ammeter', 'UNKNOWN_ID_4', 'CIRCUIT BREAKER', 'CURRENT TRANSFORMER', 'Circuit Breaker 2', 'Contractor 1', 'DIGITAL METER', 'Digital Power Meter', 'EARTH LEAKAGE RELAY', 'EMS', 'Earth Fault Relay', 'FUSE 1', 'FUSE 2', 'HRC FUSE WITH BLOWN FUSE INDICATOR', 'ISOLATOR', 'MAXIMUM DEMAND AMMETER', 'Over current Relay', 'PHASE INDICATOR LIGHTS', 'Power Quality Meter', 'RCCB', 'SELECTOR SWITCH', 'SHUNT TRIP', 'SINGLE PHASE UNFUSED TAP OFF UNIT', 'SURGE ARRESTOR 1', 'UNKNOWN_ID_27', 'TIME DELAY RELAY', 'VOLTMETER', 'ZERO PHASE SEQUENCE CURRENT TRANSFORMER']


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 3 — Build model from config + load checkpoint
# ══════════════════════════════════════════════════════
import sys
sys.path.append('/content/D-FINE')

import torch
import torch.nn as nn

# Reconstructed from your training log's resolved cfg (D-FINE-X / HGNetv2-B5).
# If you still have your ORIGINAL custom training yaml (the one passed to
# `train.py -c ...`), upload it instead and set CONFIG_PATH to that file —
# that guarantees an exact architecture match rather than a reconstruction.
RECONSTRUCTED_CFG = f'''
task: detection
model: DFINE
postprocessor: DFINEPostProcessor
num_classes: {NUM_CLASSES}
eval_spatial_size: [{RESOLUTION}, {RESOLUTION}]
use_focal_loss: True

DFINE:
  backbone: HGNetv2
  encoder: HybridEncoder
  decoder: DFINETransformer

HGNetv2:
  name: B5
  return_idx: [1, 2, 3]
  freeze_stem_only: True
  freeze_at: 0
  freeze_norm: True
  pretrained: False

HybridEncoder:
  in_channels: [512, 1024, 2048]
  feat_strides: [8, 16, 32]
  hidden_dim: 384
  use_encoder_idx: [2]
  num_encoder_layers: 1
  nhead: 8
  dim_feedforward: 2048
  dropout: 0.0
  enc_act: 'gelu'
  expansion: 1.0
  depth_mult: 1
  act: 'silu'

DFINETransformer:
  feat_channels: [384, 384, 384]
  feat_strides: [8, 16, 32]
  hidden_dim: 256
  num_levels: 3
  num_layers: 6
  eval_idx: -1
  num_queries: 300
  num_denoising: 100
  label_noise_ratio: 0.5
  box_noise_scale: 1.0
  layer_scale: 1
  num_points: [3, 6, 3]
  cross_attn_method: default
  query_select_method: default
  reg_max: 32
  reg_scale: 8

DFINEPostProcessor:
  num_top_queries: 300
'''

CONFIG_PATH = '/content/dfine_inference_config.yml'
with open(CONFIG_PATH, 'w') as f:
    f.write(RECONSTRUCTED_CFG)

from src.core import YAMLConfig

cfg = YAMLConfig(CONFIG_PATH, resume=DFINE_WEIGHTS_PATH)

checkpoint = torch.load(DFINE_WEIGHTS_PATH, map_location='cpu', weights_only=False)
if 'ema' in checkpoint:
    state = checkpoint['ema']['module']
elif 'model' in checkpoint:
    state = checkpoint['model']
else:
    state = checkpoint  # assume raw state_dict

cfg.model.load_state_dict(state)

class DFINEInferModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = cfg.model.deploy()
        self.postprocessor = cfg.postprocessor.deploy()

    def forward(self, images, orig_target_sizes):
        outputs = self.model(images)
        outputs = self.postprocessor(outputs, orig_target_sizes)
        return outputs  # (labels, boxes, scores)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DFINEInferModel(cfg).to(device).eval()
print(f'Device: {device}')
print('D-FINE model loaded from', DFINE_WEIGHTS_PATH)


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.

In [ ]:
def draw_detections(pil_image, detections):
    if len(detections) == 0:
        return pil_image

    img_np = np.array(pil_image)
    img_np = box_annotator.annotate(img_np, detections)   # thin colored boxes
    img = Image.fromarray(img_np).convert('RGBA')

    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    font = ImageFont.load_default()

    for box, cid, score in zip(detections.xyxy, detections.class_id, detections.confidence):
        cid = int(cid)
        color = PALETTE.colors[cid % len(PALETTE.colors)]
        name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else f'class_{cid}'
        text = f'{name} {score:.2f}'

        x1, y1 = box[0], box[1]
        tw, th = draw.textbbox((0, 0), text, font=font)[2:]
        pad = 2
        bg_top = max(0, y1 - th - 2 * pad)
        draw.rectangle([x1, bg_top, x1 + tw + 2 * pad, y1], fill=(color.r, color.g, color.b, 90))  # light/translucent
        draw.text((x1 + pad, bg_top + pad), text, fill=(20, 20, 20, 230), font=font)

    return Image.alpha_composite(img, overlay).convert('RGB')

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 5 — Upload test image
# ══════════════════════════════════════════════════════
from google.colab import files
from IPython.display import display
import io

print('Upload your SLD image (JPG or PNG)...')
uploaded   = files.upload()
img_name   = list(uploaded.keys())[0]
test_image = Image.open(io.BytesIO(uploaded[img_name])).convert('RGB')
W, H       = test_image.size
print(f'Image: {img_name}  |  {W} x {H} px')
thumb = test_image.copy()
thumb.thumbnail((900, 900))
display(thumb)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 6 — MODE 1: Tiled 5x5 — D-FINE
# ══════════════════════════════════════════════════════
GRID_SIZE_5 = 5
W, H = test_image.size
tiles5 = tile_image(test_image, grid_size=GRID_SIZE_5, overlap=OVERLAP)
print(f'Total tiles: {len(tiles5)} ({GRID_SIZE_5}x{GRID_SIZE_5}, {int(OVERLAP*100)}% overlap)\n')

print('--- D-FINE 5x5 ---')
tile_results_dfine5 = []
for i, (tile, x_off, y_off) in enumerate(tiles5):
    det = run_inference_dfine(tile)
    tile_results_dfine5.append((det, x_off, y_off))
    print(f'  Tile {i+1:02d}/{len(tiles5)} at ({x_off},{y_off}) => {len(det)} detections')
det_dfine_5x5 = merge_detections(tile_results_dfine5, W, H)
print(f'After NMS: {len(det_dfine_5x5)} detections')
for i,(box,score,cls) in enumerate(zip(det_dfine_5x5.xyxy, det_dfine_5x5.confidence, det_dfine_5x5.class_id)):
    name = CLASS_NAMES[int(cls)] if int(cls) < len(CLASS_NAMES) else f'class_{cls}'
    print(f'  [{i+1}] {name}  conf={score:.3f}  box=[{int(box[0])},{int(box[1])},{int(box[2])},{int(box[3])}]')

result_dfine_5x5 = draw_detections(test_image, det_dfine_5x5)
out = result_dfine_5x5.copy(); out.thumbnail((1200,1200))
print('\nD-FINE 5x5 result:'); display(out)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 7 — MODE 2: Tiled 4x4 — D-FINE
# ══════════════════════════════════════════════════════
W, H = test_image.size
tiles4 = tile_image(test_image, grid_size=GRID_SIZE, overlap=OVERLAP)
print(f'Total tiles: {len(tiles4)} ({GRID_SIZE}x{GRID_SIZE}, {int(OVERLAP*100)}% overlap)\n')

print('--- D-FINE 4x4 ---')
tile_results_dfine4 = []
for i, (tile, x_off, y_off) in enumerate(tiles4):
    det = run_inference_dfine(tile)
    tile_results_dfine4.append((det, x_off, y_off))
    print(f'  Tile {i+1:02d}/{len(tiles4)} at ({x_off},{y_off}) => {len(det)} detections')
det_dfine_4x4 = merge_detections(tile_results_dfine4, W, H)
print(f'After NMS: {len(det_dfine_4x4)} detections')
for i,(box,score,cls) in enumerate(zip(det_dfine_4x4.xyxy, det_dfine_4x4.confidence, det_dfine_4x4.class_id)):
    name = CLASS_NAMES[int(cls)] if int(cls) < len(CLASS_NAMES) else f'class_{cls}'
    print(f'  [{i+1}] {name}  conf={score:.3f}  box=[{int(box[0])},{int(box[1])},{int(box[2])},{int(box[3])}]')

result_dfine_4x4 = draw_detections(test_image, det_dfine_4x4)
out = result_dfine_4x4.copy(); out.thumbnail((1200,1200))
print('\nD-FINE 4x4 result:'); display(out)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 8 — Save outputs
# ══════════════════════════════════════════════════════
base = img_name.rsplit('.', 1)[0]

paths = {
    f'{base}_dfine_5x5.jpg' : result_dfine_5x5,
    f'{base}_dfine_4x4.jpg' : result_dfine_4x4,
}

for fname, img in paths.items():
    p = f'/content/{fname}'
    img.save(p, quality=95)
    print(f'Saved: {p}')

print('\nDownload all from Files panel (left sidebar).')
